In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ziyadaltalhi/youtube-data/YouTube Datasets - After Processing/المنطقة الوسطى/Youtube Data/القصيم/dataset_youtube-comments-scraper_2025-11-13_07-31-25-109_textready_analysis.xlsx
/kaggle/input/datasets/ziyadaltalhi/youtube-data/YouTube Datasets - After Processing/المنطقة الوسطى/Youtube Data/القصيم/dataset_youtube-comments-scraper_2025-11-13_07-06-14-938_textready_cleaned.xlsx
/kaggle/input/datasets/ziyadaltalhi/youtube-data/YouTube Datasets - After Processing/المنطقة الوسطى/Youtube Data/القصيم/dataset_youtube-comments-scraper_2025-11-11_07-18-20-122_textready_cleaned.xlsx
/kaggle/input/datasets/ziyadaltalhi/youtube-data/YouTube Datasets - After Processing/المنطقة الوسطى/Youtube Data/القصيم/dataset_youtube-comments-scraper_2025-11-11_08-03-55-015_textready_cleaned.xlsx
/kaggle/input/datasets/ziyadaltalhi/youtube-data/YouTube Datasets - After Processing/المنطقة الوسطى/Youtube Data/القصيم/dataset_youtube-comments-scraper_2025-11-11_07-55-12-414_textready_cleaned.xlsx

In [3]:
import os, glob, re, pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix,f1_score
from sklearn.utils import resample

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D, GlobalMaxPooling1D
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [4]:
# =========================================================
# (0) PATH + SETTINGS  (ALL REGIONS) + QUICK STRUCTURE CHECK
# =========================================================
ALL_REGIONS_ROOT = "/kaggle/input/datasets/ziyadaltalhi/youtube-data/YouTube Datasets - After Processing/"
TEXT_COL = "Text_TR"  # ✅ العمود المراد العمل عليه
STAR_CANDIDATES = {"stars", "Stars"}
print("\n[STEP 0] Path checks (ALL REGIONS)")
print("Root exists:", os.path.exists(ALL_REGIONS_ROOT))
print("Root is dir:", os.path.isdir(ALL_REGIONS_ROOT))

region_dirs = sorted([d for d in glob.glob(os.path.join(ALL_REGIONS_ROOT, "*")) if os.path.isdir(d)])
print("Region folders found:", len(region_dirs))
print("Regions:", [os.path.basename(d) for d in region_dirs])

# عرض سريع لأول 3 مناطق: عدد المدن داخل كل منطقة
for rd in region_dirs[:3]:
    city_dirs = sorted([d for d in glob.glob(os.path.join(rd, "*")) if os.path.isdir(d)])
    print(f"  - {os.path.basename(rd)}: cities={len(city_dirs)} (sample: {[os.path.basename(x) for x in city_dirs[:5]]})")



[STEP 0] Path checks (ALL REGIONS)
Root exists: True
Root is dir: True
Region folders found: 5
Regions: ['المنطقة الجنوبية', 'المنطقة الشرقية', 'المنطقة الشمالية', 'المنطقة الغربية', 'المنطقة الوسطى']
  - المنطقة الجنوبية: cities=4 (sample: ['بيانات اليوتيوب لمنطقة الباحة - بعد المعالجة', 'بيانات اليوتيوب لمنطقة جازان - بعد المعالجة', 'بيانات اليوتيوب لمنطقة عسير - بعد المعالجة', 'بيانات اليوتيوب لمنطقة نجران - بعد المعالجة'])
  - المنطقة الشرقية: cities=1 (sample: ['بيانات اليوتيوب للشرقية - بعد المعالجة'])
  - المنطقة الشمالية: cities=1 (sample: ['( المنطقة الشمالية ) Youtube Data -  After Preproccesing'])


In [5]:
# =========================================================
# Helpers: extract region name (inside parentheses) + city
# =========================================================
PAREN_RE = re.compile(r"\((.*?)\)")

def extract_region_from_folder(region_folder_name: str) -> str:
    """
    يستخرج النص بين ( ) من اسم مجلد المنطقة.
    مثال: "( المنطقة الغربية ) Google Maps Data - After Cleaning" -> "المنطقة الغربية"
    إذا لم توجد أقواس يرجع اسم المجلد نفسه.
    """
    m = PAREN_RE.search(region_folder_name)
    if m:
        return m.group(1).strip()
    return region_folder_name.strip()

def list_region_folders(root_folder: str):
    region_dirs = sorted([d for d in glob.glob(os.path.join(root_folder, "*")) if os.path.isdir(d)])
    return region_dirs

def list_city_folders(region_dir: str):
    return sorted([d for d in glob.glob(os.path.join(region_dir, "*")) if os.path.isdir(d)])

def list_files_in_city(city_dir: str):
    return (
        glob.glob(os.path.join(city_dir, "*.xlsx")) +
        glob.glob(os.path.join(city_dir, "*.xls")) +
        glob.glob(os.path.join(city_dir, "*.csv"))
    )

def read_any_file(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    elif ext == ".csv":
        # ترميزات عربية شائعة
        try:
            df = pd.read_csv(path, encoding="utf-8")
        except UnicodeDecodeError:
            try:
                df = pd.read_csv(path, encoding="utf-8-sig")
            except UnicodeDecodeError:
                df = pd.read_csv(path, encoding="cp1256")
    else:
        raise ValueError(f"Unsupported extension: {ext}")
    df.columns = [str(c).strip() for c in df.columns]
    return df

def standardize_star_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    - يقبل Stars / stars / ... ويوحّدها إلى عمود اسمه 'Stars' إن وجد.
    - لا يغير القيم الآن.
    """
    colmap = {c.lower(): c for c in df.columns}
    # إذا موجود Stars بالاسم الصحيح خلاص
    if "Stars" in df.columns:
        return df

    # ابحث عن أي عمود اسمه stars case-insensitive
    if "stars" in colmap:
        df = df.rename(columns={colmap["stars"]: "Stars"})
        return df

    # مرونة إضافية: إذا فيه rating مثلا
    for cand in list(STAR_CANDIDATES):
        key = cand.lower()
        if key in colmap:
            df = df.rename(columns={colmap[key]: "Stars"})
            return df

    return df  # لم نجد عمود نجوم

In [6]:

# =========================================================
# [STEP 0] Path checks + show regions/cities counts
# =========================================================
print("\n[STEP 0] Path checks (ALL REGIONS)")
print("Root exists:", os.path.exists(ALL_REGIONS_ROOT))
print("Root is dir :", os.path.isdir(ALL_REGIONS_ROOT))

region_dirs = list_region_folders(ALL_REGIONS_ROOT)
print("Region folders found:", len(region_dirs))

# عرض أسماء المناطق بصيغة الاسم داخل الأقواس
for rd in region_dirs:
    folder_name = os.path.basename(rd)
    region_name = extract_region_from_folder(folder_name)
    print(f"- Folder: {folder_name}  --> Region: {region_name}")



[STEP 0] Path checks (ALL REGIONS)
Root exists: True
Root is dir : True
Region folders found: 5
- Folder: المنطقة الجنوبية  --> Region: المنطقة الجنوبية
- Folder: المنطقة الشرقية  --> Region: المنطقة الشرقية
- Folder: المنطقة الشمالية  --> Region: المنطقة الشمالية
- Folder: المنطقة الغربية  --> Region: المنطقة الغربية
- Folder: المنطقة الوسطى  --> Region: المنطقة الوسطى


In [7]:
import os
import pandas as pd


# =========================================================
# (0) LIST REGION FOLDERS
# =========================================================
def list_region_folders(root_folder):
    return [
        os.path.join(root_folder, d)
        for d in os.listdir(root_folder)
        if os.path.isdir(os.path.join(root_folder, d))
    ]


# =========================================================
# (0.1) EXTRACT REGION NAME
# =========================================================
def extract_region_from_folder(folder_name: str):
    return folder_name


# =========================================================
# (1) GET ALL VALID FILES (RECURSIVE)
# =========================================================
def get_analysis_files_recursive(region_path):
    files = []

    for root, dirs, filenames in os.walk(region_path):

        dirs[:] = [d for d in dirs if not d.startswith(".")]

        for fname in filenames:
            lower = fname.lower()

            if (
                lower.endswith((".xlsx", ".xls")) and
                "analysis" in lower and
                "summary" not in lower
            ):
                full_path = os.path.join(root, fname)
                files.append(full_path)

    return sorted(files)


# =========================================================
# (2) SCAN AVAILABILITY
# =========================================================
def scan_availability(root_folder: str):
    rows = []

    for rd in list_region_folders(root_folder):
        region_folder = os.path.basename(rd)
        region_name = extract_region_from_folder(region_folder)

        files = get_analysis_files_recursive(rd)

        rows.append({
            "Region_Folder": region_folder,
            "Region_Name": region_name,
            "Files": len(files),
            "Sample_File": files[0] if files else None
        })

    return pd.DataFrame(rows).sort_values("Files", ascending=False).reset_index(drop=True)


# =========================================================
# (3) READ ALL REGIONS (FIXED)
# =========================================================
def read_all_regions_chunked(root_folder: str, max_files_per_region=None, keep_columns=None):

    all_frames = []
    report_rows = []

    region_dirs = list_region_folders(root_folder)

    for r_idx, rd in enumerate(region_dirs, 1):
        region_folder = os.path.basename(rd)
        region_name = extract_region_from_folder(region_folder)

        region_files = get_analysis_files_recursive(rd)

        if max_files_per_region is not None:
            region_files = region_files[:max_files_per_region]

        print(f"\n[STEP 1] Region {r_idx}/{len(region_dirs)}: {region_name} | files={len(region_files)}")

        n_rows_region = 0
        has_text = 0
        has_stars = 0
        read_ok = 0
        read_err = 0

        for i, p in enumerate(region_files, 1):
            try:
                df = read_any_file(p)
                df = standardize_star_column(df)

                # ===============================
                # SAFE CITY / REGION DETECTION
                # ===============================
                abs_path = os.path.abspath(p)
                city_name = os.path.basename(os.path.dirname(abs_path))

                if city_name.startswith("Tik_"):
                    city_name = os.path.basename(os.path.dirname(os.path.dirname(abs_path)))

                # ===============================
                # METADATA
                # ===============================
                df["Region_Folder"] = region_folder
                df["Region_Name"] = region_name
                df["City_Folder"] = city_name
                df["Source_File"] = os.path.basename(p)
                df["__path__"] = p

                # ===============================
                # CHECK COLUMNS
                # ===============================
                if TEXT_COL in df.columns:
                    has_text += 1
                if "Stars" in df.columns:
                    has_stars += 1

                # ===============================
                # KEEP ONLY SELECTED COLUMNS
                # ===============================
                if keep_columns is not None:
                    keep = [c for c in keep_columns if c in df.columns]
                    meta = ["Region_Folder", "Region_Name", "City_Folder", "Source_File", "__path__"]
                    keep = list(dict.fromkeys(keep + meta))
                    df = df[keep].copy()

                n_rows_region += len(df)
                all_frames.append(df)
                read_ok += 1

                if i <= 2:
                    print(f"  sample file [{i}] {region_name}/{city_name}/{os.path.basename(p)} rows={len(df)}")

                if i % 50 == 0:
                    print(f"  progress: {i}/{len(region_files)} files")

            except Exception as e:
                read_err += 1
                print(f"❌ Error reading: {p}")
                print("ERROR TYPE:", type(e).__name__)
                print("ERROR:", repr(e))

        report_rows.append({
            "Region_Name": region_name,
            "Region_Folder": region_folder,
            "Files_Read_OK": read_ok,
            "Files_Read_Err": read_err,
            "Files_with_Text": has_text,
            "Files_with_Stars": has_stars,
            "Total_Rows_This_Region": n_rows_region
        })

    df_region_report = pd.DataFrame(report_rows) \
        .sort_values("Total_Rows_This_Region", ascending=False) \
        .reset_index(drop=True)

    df_all = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()

    return df_all, df_region_report


# =========================================================
# (4) RUN
# =========================================================
print("\n[STEP 1A] Availability per region:")
df_av = scan_availability(ALL_REGIONS_ROOT)
display(df_av)

KEEP_COLS = [
    TEXT_COL,
    "Stars",
]

df1, df_region_report = read_all_regions_chunked(
    ALL_REGIONS_ROOT,
    max_files_per_region=None,
    keep_columns=KEEP_COLS
)

print("\n[STEP 1B] Column availability per region:")
display(df_region_report)

print("\n[STEP 1 RESULT] df1 shape:", df1.shape)
print("Columns:", df1.columns.tolist())


[STEP 1A] Availability per region:


,Region_Folder,Region_Name,Files,Sample_File
0,المنطقة الجنوبية,المنطقة الجنوبية,82,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
1,المنطقة الوسطى,المنطقة الوسطى,47,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
2,المنطقة الشرقية,المنطقة الشرقية,46,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
3,المنطقة الشمالية,المنطقة الشمالية,20,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
4,المنطقة الغربية,المنطقة الغربية,3,/kaggle/input/datasets/ziyadaltalhi/youtube-da...



[STEP 1] Region 1/5: المنطقة الوسطى | files=47
  sample file [1] المنطقة الوسطى/الرياض/dataset_youtube-comments-scraper_2025-11-11_05-58-34-900_textready_analysis.xlsx rows=192
  sample file [2] المنطقة الوسطى/الرياض/dataset_youtube-comments-scraper_2025-11-11_06-00-40-626_textready_analysis.xlsx rows=19

[STEP 1] Region 2/5: المنطقة الجنوبية | files=82
  sample file [1] المنطقة الجنوبية/بيانات اليوتيوب لمنطقة الباحة - بعد المعالجة/Al-Baha Vedio Comments 10_textready_analysis.xlsx rows=62
  sample file [2] المنطقة الجنوبية/بيانات اليوتيوب لمنطقة الباحة - بعد المعالجة/Al-Baha Vedio Comments 11_textready_analysis.xlsx rows=59
  progress: 50/82 files

[STEP 1] Region 3/5: المنطقة الشمالية | files=20
  sample file [1] المنطقة الشمالية/العلا/العلا2_textready_analysis.xlsx rows=21
  sample file [2] المنطقة الشمالية/العلا/العلا3_textready_analysis.xlsx rows=862

[STEP 1] Region 4/5: المنطقة الشرقية | files=46
  sample file [1] المنطقة الشرقية/الاحساء/10_textready_analysis.xlsx rows=4
  sampl

,Region_Name,Region_Folder,Files_Read_OK,Files_Read_Err,Files_with_Text,Files_with_Stars,Total_Rows_This_Region
0,المنطقة الجنوبية,المنطقة الجنوبية,82,0,82,82,9240
1,المنطقة الوسطى,المنطقة الوسطى,47,0,47,47,5097
2,المنطقة الشمالية,المنطقة الشمالية,20,0,20,20,4499
3,المنطقة الشرقية,المنطقة الشرقية,46,0,46,46,387
4,المنطقة الغربية,المنطقة الغربية,3,0,3,3,216



[STEP 1 RESULT] df1 shape: (19439, 7)
Columns: ['Text_TR', 'Stars', 'Region_Folder', 'Region_Name', 'City_Folder', 'Source_File', '__path__']


In [8]:
df1['Stars'].value_counts()

Stars
5    9930
1    6458
3    3051
Name: count, dtype: int64

In [9]:
df1

,Text_TR,Stars,Region_Folder,Region_Name,City_Folder,Source_File,__path__
0,طالما فيها ال سعود الاذلاء رح تضلها بلاد ذليلة,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
1,تعال لنا ثاني مره نتشرف فيك,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
2,يامرحبا فيك عيد الزياره واجلس اكثر وروح لكذا م...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
3,المسمار هو القرنفل وليس جوزة الطيب,3,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
4,اهلا اهلا فيك في بلدنا منور يا وحش,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
...,...,...,...,...,...,...,...
19434,وعليكم السلام ورحمة الله الحمد الله يبارك فيك ...,5,المنطقة الغربية,المنطقة الغربية,مكة,مكة مول_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
19435,بما انك دعيت هذه الدهوة اقول الله يعينك,1,المنطقة الغربية,المنطقة الغربية,مكة,مكة مول_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
19436,انا متاكد ان سلمان سيلبي طلبك بكل سرور ولذلك و...,5,المنطقة الغربية,المنطقة الغربية,مكة,مكة مول_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
19437,وفقك الله اختي، نقول توكلت علي الله قال رجل لل...,5,المنطقة الغربية,المنطقة الغربية,مكة,مكة مول_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/youtube-da...


In [10]:
# =========================================================
# (2) FILTER ONLY TRULY-EMPTY Text_TR (NaN + "" + "nan" + "[]" ...)
# =========================================================
print(f"\n[STEP 2] Using {TEXT_COL} + strong empty filter")

# ✅ الداتا الجديدة
if TEXT_COL not in df1.columns:
    raise ValueError(f"عمود {TEXT_COL} غير موجود في البيانات.")

before2 = len(df1)

# لا نحول إلى string الآن (مهم)
s = df1[TEXT_COL]

empty_like = {
    "", "nan", "NaN", "none", "None", "NONE",
    "<NA>", "[]", "[ ]", "{}", "null", "NULL"
}

empty_mask = (
    s.isna() |
    s.astype(str).str.strip().isin(empty_like)
)

empty_count = int(empty_mask.sum())

# ✅ الناتج الجديد
df2 = df1.loc[~empty_mask].copy()

# العمود الذي سيستخدم لاحقاً للمودل
df2["TEXT_FOR_MODEL"] = df2[TEXT_COL].astype(str).str.strip()

print("\n[STEP 2 RESULT]")
print("Rows before:", before2)
print("Empty-like rows:", empty_count)
print("Rows after:", len(df2))

print("\nSample empty-like values:")
print(s.loc[empty_mask].head(10).astype(str).tolist())


[STEP 2] Using Text_TR + strong empty filter

[STEP 2 RESULT]
Rows before: 19439
Empty-like rows: 0
Rows after: 19439

Sample empty-like values:
[]


In [11]:
# =========================================================
# (4) SAFE TEXT CLEANING (INDEPENDENT)
#   - لا يعتمد على Stars_num أو y_bin
#   - يحافظ على الأعمدة التعريفية
# =========================================================

import re
import pandas as pd

print("\n[STEP 4] Safe Cleaning (Independent)")

TEXT_COL = "Text_TR"

if TEXT_COL not in df2.columns:
    raise ValueError(f"{TEXT_COL} غير موجود في df2")

# =========================
# Regex definitions
# =========================
AR_DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670]")
AR_TATWEEL_RE    = re.compile(r"\u0640")

# =========================
# Safe Arabic normalization
# =========================
def normalize_arabic_safe(text: str) -> str:
    text = AR_DIACRITICS_RE.sub("", text)   # remove diacritics
    text = AR_TATWEEL_RE.sub("", text)      # remove tatweel (ـ)
    text = re.sub(r"[إأآا]", "ا", text)      # unify alef only
    return text

# =========================
# Main cleaning function
# =========================
def clean_text(s):
    if pd.isna(s):
        return ""
    
    s = str(s).strip().lower()
    if not s:
        return ""

    # remove links / emails / mentions / hashtags
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"\S+@\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    s = re.sub(r"#\w+", " ", s)

    # remove emojis
    s = re.sub(r"[\U00010000-\U0010ffff]", " ", s)

    # keep Arabic / English / digits / spaces
    s = re.sub(r"[^0-9a-z\u0600-\u06FF\s]", " ", s)

    # safe normalize
    s = normalize_arabic_safe(s)

    # reduce repeated letters (جمييييل -> جمييل)
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)

    # collapse spaces
    s = re.sub(r"\s+", " ", s).strip()

    # remove if numbers only
    if re.fullmatch(r"\d+", s):
        return ""

    return s


# =========================
# Preserve metadata columns if exist
# =========================
META_COLS = ["Stars","Region_Name", "Region_Folder", "City_Folder", "Source_File", "__path__","CategoryName"]
meta_existing = [c for c in META_COLS if c in df2.columns]

# =========================
# BEFORE dataframe
# =========================
df_before_clean = df2[[TEXT_COL] + meta_existing].copy()
df_before_clean = df_before_clean.rename(columns={TEXT_COL: "text_before"})

display(df_before_clean.head(10))


# =========================
# Apply cleaning
# =========================
df4 = df2.copy()
df4["text_clean"] = df4[TEXT_COL].apply(clean_text)

# =========================
# Compare dataframe
# =========================
df_compare = df4[[TEXT_COL, "text_clean"] + meta_existing].copy()
df_compare = df_compare.rename(columns={TEXT_COL: "text_before"})

display(df_compare.head(20))


# =========================
# Removed rows (empty after clean)
# =========================
df_removed = df_compare[df_compare["text_clean"].str.len() == 0].copy()

print("Removed rows (empty after clean):", len(df_removed))
display(df_removed.head(20))


# =========================
# Final cleaned dataframe
# =========================
df4 = df_compare[df_compare["text_clean"].str.len() > 0].copy()

print("Final cleaned shape:", df4.shape)
print("Columns:", df4.columns.tolist())

display(df4.head(20))


# =========================
# Safety check examples
# =========================
print("\n[CHECK] safe normalization examples:")
for t in ["سيئ", "سيئة", "سيء", "المكان سيئ جدا", "المكان رائع جدا", "12345", "مطعم 123"]:
    print(f"{t}  ->  {clean_text(t)}")


[STEP 4] Safe Cleaning (Independent)


,text_before,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__
0,طالما فيها ال سعود الاذلاء رح تضلها بلاد ذليلة,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
1,تعال لنا ثاني مره نتشرف فيك,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
2,يامرحبا فيك عيد الزياره واجلس اكثر وروح لكذا م...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
3,المسمار هو القرنفل وليس جوزة الطيب,3,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
4,اهلا اهلا فيك في بلدنا منور يا وحش,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
5,هذا تعليق ساجعله ان شاء الله صدقة جارية لي ولك...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
6,ياريت ترجع تزور اليمن احس مااعطيتها حقها واشوف...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
7,اطلع برج القاهره في مصر من تحتك اطول نهر في ال...,3,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
8,اول مره اعرف عن المكان ذا بحياتي,3,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
9,فيديو رائع وجميل ومعلومات مفيدة جدا,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...


,text_before,text_clean,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__
0,طالما فيها ال سعود الاذلاء رح تضلها بلاد ذليلة,طالما فيها ال سعود الاذلاء رح تضلها بلاد ذليلة,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
1,تعال لنا ثاني مره نتشرف فيك,تعال لنا ثاني مره نتشرف فيك,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
2,يامرحبا فيك عيد الزياره واجلس اكثر وروح لكذا م...,يامرحبا فيك عيد الزياره واجلس اكثر وروح لكذا م...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
3,المسمار هو القرنفل وليس جوزة الطيب,المسمار هو القرنفل وليس جوزة الطيب,3,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
4,اهلا اهلا فيك في بلدنا منور يا وحش,اهلا اهلا فيك في بلدنا منور يا وحش,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
5,هذا تعليق ساجعله ان شاء الله صدقة جارية لي ولك...,هذا تعليق ساجعله ان شاء الله صدقة جارية لي ولك...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
6,ياريت ترجع تزور اليمن احس مااعطيتها حقها واشوف...,ياريت ترجع تزور اليمن احس مااعطيتها حقها واشوف...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
7,اطلع برج القاهره في مصر من تحتك اطول نهر في ال...,اطلع برج القاهره في مصر من تحتك اطول نهر في ال...,3,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
8,اول مره اعرف عن المكان ذا بحياتي,اول مره اعرف عن المكان ذا بحياتي,3,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
9,فيديو رائع وجميل ومعلومات مفيدة جدا,فيديو رائع وجميل ومعلومات مفيدة جدا,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...


Removed rows (empty after clean): 0


,text_before,text_clean,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__


Final cleaned shape: (19439, 8)
Columns: ['text_before', 'text_clean', 'Stars', 'Region_Name', 'Region_Folder', 'City_Folder', 'Source_File', '__path__']


,text_before,text_clean,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__
0,طالما فيها ال سعود الاذلاء رح تضلها بلاد ذليلة,طالما فيها ال سعود الاذلاء رح تضلها بلاد ذليلة,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
1,تعال لنا ثاني مره نتشرف فيك,تعال لنا ثاني مره نتشرف فيك,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
2,يامرحبا فيك عيد الزياره واجلس اكثر وروح لكذا م...,يامرحبا فيك عيد الزياره واجلس اكثر وروح لكذا م...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
3,المسمار هو القرنفل وليس جوزة الطيب,المسمار هو القرنفل وليس جوزة الطيب,3,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
4,اهلا اهلا فيك في بلدنا منور يا وحش,اهلا اهلا فيك في بلدنا منور يا وحش,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
5,هذا تعليق ساجعله ان شاء الله صدقة جارية لي ولك...,هذا تعليق ساجعله ان شاء الله صدقة جارية لي ولك...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
6,ياريت ترجع تزور اليمن احس مااعطيتها حقها واشوف...,ياريت ترجع تزور اليمن احس مااعطيتها حقها واشوف...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
7,اطلع برج القاهره في مصر من تحتك اطول نهر في ال...,اطلع برج القاهره في مصر من تحتك اطول نهر في ال...,3,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
8,اول مره اعرف عن المكان ذا بحياتي,اول مره اعرف عن المكان ذا بحياتي,3,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
9,فيديو رائع وجميل ومعلومات مفيدة جدا,فيديو رائع وجميل ومعلومات مفيدة جدا,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...



[CHECK] safe normalization examples:
سيئ  ->  سيئ
سيئة  ->  سيئة
سيء  ->  سيء
المكان سيئ جدا  ->  المكان سيئ جدا
المكان رائع جدا  ->  المكان رائع جدا
12345  ->  
مطعم 123  ->  مطعم 123


In [12]:
import re

# =========================
# قاموس الجوانب (Arabic-only) - موسع
# =========================
ASPECT_PATTERNS = {

    # 1) النظافة
    "النظافة": [
        # كلمات مباشرة مع حدود الكلمات
        r"\b(نظافه|نظافة)\b",
        r"\bنظافه\s+(عامه|المكان|المحل|المطعم|الفرع)\b",
        r"\b(نظيف|نظيفه|نظيفة|نظيفين|نظيفون|نظيفات)\b",
        r"\bنظيف(ه)?\s+(جدا|مره|للغايه|للغاية)\b",
        r"\b(متسخ|متسخه|متسخة|متسخين|متسخات)\b",
        r"\b(غير|مو|مش|ما)\s+نظيف(ه)?\b",
        
        # قذارة/وسخ/وصخ/زبالة
        r"\b(وسخ|وسخه|وسخة|وسخان|وسخين|وسخات)\b",
        r"\b(وصخ|وصخه|وصخة|وصخان|وصخين)\b",
        r"\b(قذر|قذره|قذرة|قذرين|قذارات|قذاره|قذارة)\b",
        r"\b(زباله|زبالة|قمامه|قمامة|نفايات|مخلفات)\b",
        r"\bزباله\s+بالارض\b",
        r"\bقمامه\s+موجوده\b",
        
        # تعقيم/روائح مرتبطة بالنظافة
        r"\b(تعقيم|معقم|تطهير|مطهر|منظف|كلور)\b",
        r"\bريح(ه|ة)\s+(وصخه|كريهه|كريهة)\b",
        r"\bريح(ه|ة)\s+مو\s+زينه\b",
        r"\b(روائح\s+كريهه|زفاره|نتن|معفن)\b",
        
        # أرضيات/طاولات/مقاعد
        r"\b(ارضيه|ارضية|ارضيات|الارض|الارضيه|الارضية)\b",
        r"\b(طاولات|الطاولات|كراسي|المقاعد)\s+(وصخه|وسخه|وصخة|وسخة|متسخه|متسخة)\b",
        
        # حشرات كجزء من النظافة
        r"\b(حشره|حشرة|حشرات|نمل|ذباب|صراصير|صرصور)\b"
    ],

    # 2) دورات المياه
    "دورات المياه": [
        r"\bدورات\s+المياه\b",
        r"\bدور(ه|ة)\s+(مياه|المياه|الميه|ميه)\b",
        r"\b(حمام|الحمام|حمامات|دورات|توليت|تواليت|مرحاض|مراحيض)\b",
        
        # توفر/صلاحية
        r"\b(ما|مافي|مو|مش)\s+في\s+(حمام|دورات)\b",
        r"\bالحمام\s+(مقفول|مقفل|خربان)\b",
        r"\b(المرحاض|السيفون|المغسله|المغسلة|مغسله|مغسلة)\s+خربان(ه|ة)?\b",
        
        # مياه/صنابير/مغاسل
        r"\b(مويه|ماء|مياه|حنفيه|حنفية|صنبور|مغسله|مغسلة|مغاسل)\b",
        r"\bمغسل(ه|ة)\s+يدين\b",
        r"\bمكان\s+غسيل\b",
        
        # مستلزمات الحمام
        r"\b(صابون|معقم|مناديل|محارم|منشفه|منشفة|مجفف)\b",
        r"\bورق\s+تواليت\b",
        
        # روائح/اتساخ داخل الحمام
        r"\b(حمام|دورات|الحمامات)\s+(وسخ|وصخ|قذر|متسخ|وسخه|وصخه|قذره|متسخه)\b",
        r"\b(ريح(ه|ة)|زفاره|روائح)\s+(الحمام|كريهه)\b",
        r"\bالحمام\s+ريحته\s+كريهه\b",
        
        # ازدحام/حجم
        r"\b(زحم(ه|ة)|طابور|انتظار)\s+الحمام\b",
        r"\bحمام\s+ضيق\b",
        r"\bحمامات\s+قليله\b"
    ],

    # 3) الخدمة
    "الخدمة": [
        r"\b(خدمه|خدمة|الخدمه|الخدمة)\b",
        r"\bمستوى\s+الخدم(ه|ة)\b",
        r"\b(جود(ه|ة)|نوعي(ه|ة))\s+الخدم(ه|ة)\b",
        r"\bخدم(ه|ة)\s+العملاء\b",
        
        r"\b(تعامل|التعامل|اسلوب|الاسلوب|معامله|معاملة|المعامله|المعاملة)\b",
        r"\bطريق(ه|ة)\s+التعامل\b",
        r"\b(التجاوب|الاهتمام)\b",
        
        r"\b(استقبال|الاستقبال|ترحيب|الترحيب)\b",
        
        r"\b(موظف|موظفين|موظفون|العامل|العمال|الطاقم)\b",
        r"\b(الكاشير|الكاشيره|الكاشيرة|الصراف|المحاسب)\b",
        
        r"\b(مدير|المدير|الاداره|الادارة|المشرف|مسؤول|مسئول)\b",
        
        # صفات إيجابية
        r"\b(متعاون|متعاونين|متعاونون|تعاون|لبق|محترم|احترام)\b",
        r"\b(رايق|را[يى]ق|بشوش|مبتسم)\b",
        
        # صفات سلبية
        r"\b(وقح|وقاحه|وقاحة)\b",
        r"\bقل(ه|ة)\s+ادب\b",
        r"\bعدم\s+احترام\b",
        r"\b(سيء|سوء)\s+التعامل\b",
        r"\bاسلوب\s+سيء\b",
        r"\b(تجاهل|يتجاهل|يتجاهلون)\b",
        
        # الرد والتواصل
        r"\b(ما|مو|مش)\s+يرد(ون)?\b",
        r"\bيردون\s+ببطء\b",
        r"\bرد\s+(متاخر|متأخر|بطيء|بطئ)\b",
        
        # تقييم الخدمة
        r"\bخدم(ه|ة)\s+(سيئه|سيئة|ضعيفه|ضعيفة|ممتازه|ممتازة|رائعه|رائعة|سريعه|سريعة|بطيئه|بطيئة)\b"
    ],

    # 4) مواقف السيارات
    "مواقف السيارات": [
        r"\b(مواقف|موقف)\b",
        r"\bمواقف\s+السيارات\b",
        r"\bموقف\s+(السياره|السيارة|السيارات)\b",
        r"\b(باركنج|بارك[يى]نج)\b",
        
        r"\bمواقف\s+(قليله|قليلة|محدوده|محدودة)\b",
        r"\b(ما|مافي|مو|مش)\s+في\s+مواقف\b",
        r"\bبدون\s+مواقف\b",
        
        r"\bصعب\s+(تلقى|احصل|نلقى)\s+موقف\b",
        r"\b(تدوير|لفات)\s+على\s+موقف\b",
        
        r"\b(زحم(ه|ة)|ازدحام)\s+(مواقف|المواقف)\b",
        r"\bالمواقف\s+(زحمه|زحمة|مكتظه|مكتظة)\b",
        
        r"\bمواقف\s+(بعيده|بعيدة|قريبه|قريبة|مظلمه|مظلمة)\b",
        r"\bمواقف\s+غير\s+(مريحه|مريحة|مرتبه|مرتبة|منظمه|منظمة)\b",
        r"\bمواقف\s+(ترابيه|ترابية)\b",
        r"\bتنظيم\s+المواقف\b",
        
        r"\bمواقف\s+(مخصصه|مخصصة|خاصه|خاصة)\b",
        r"\bمواقف\s+(للعائلات|لذوي\s+الاحتياجات)\b"
    ],

    # 5) الانتظار
    "الانتظار": [
        r"\b(انتظار|الانتظار)\b",
        r"\bوقت\s+انتظار\b",
        r"\b(مد(ه|ة)|فتر(ه|ة))\s+انتظار\b",
        r"\b(طابور|صف|الدور)\b",
        
        r"\b(تاخير|تأخير|تتاخر|تتأخر|يتاخر|يتأخر|متاخر|متأخر)\b",
        
        r"\b(ياخذ|يأخذ)\s+(وقت|وقته)\b",
        r"\b(طول|طويل)\s+وقت\b",
        r"\bوقت\s+طويل\b",
        r"\b(طولنا|تطويل)\b",
        
        r"\b(بطيء|بطئ|بطيئه|بطيئة)\b",
        r"\bبط[يى]ء(\s+جدا)?\b",
        
        r"\b(سرعه|سرعة|سريع|سريعه|سريعة|سريعين|سريعون)\b",
        r"\b(فوري|فورية)\b",
        r"\bبدون\s+انتظار\b",
        
        r"\b(تجهيز|تحضير)\b",
        r"\bتجهيز\s+(بطيء|بطئ|سريع)\b",
        r"\bتاخير\s+التجهيز\b"
    ],

    # 6) الأسعار
    "الأسعار": [
        r"\b(سعر|اسعار|الاسعار|تسعيره|تسعيرة|التسعيره|التسعيرة)\b",
        r"\b(قيمه|قيمة|القيمه|القيمة)\b",
        r"\bقيم(ه|ة)\s+(مقابل|السعر)\b",
        
        r"\b(غالي|غاليه|غالية|غاليين|غاليون)\b",
        r"\bغال[يى](\s+(جدا|مره))?\b",
        r"\b(مرتفع|مرتفعه|مرتفعة)\b",
        
        r"\bمبالغ\s+في(ه|ها)?\b",
        r"\b(مبالغه|مبالغة|استغلال)\b",
        r"\b(ينهبون|ينهب|غلاء)\b",
        
        r"\b(رخيص|رخيصه|رخيصة|رخيصين|رخيصون)\b",
        r"\bرخيص(\s+(جدا|مره))?\b",
        r"\b(مناسب|مناسبه|مناسبة)\b",
        r"\bاسعار\s+مناسب(ه|ة)\b",
        
        r"\b(يستاهل|يستحق)\b",
        r"\b(ما|مو|مش)\s+(يستاهل|يسوى)\b",
        r"\bقيم(ه|ة)\s+(ممتازه|ممتازة|جيده|جيدة)\b",
        
        r"\b(عروض|خصم|تخفيض|تخفيضات)\b",
        r"\bاسعار\s+العروض\b"
    ],

    # 7) الطعام
    "الطعام": [
        r"\b(اكل|الاكل|طعام|الطعام)\b",
        r"\b(وجبه|وجبة|وجبات|صحن|اطباق|طبق)\b",
        
        r"\b(طعم|طعمه|طعمة|مذاق|نكهه|نكهة)\b",
        
        r"\b(لذيذ|لذيذه|لذيذة)\b",
        r"\bلذيذ(\s+(جدا|مره))?\b",
        r"\b(شهي|يشهي|شهية)\b",
        r"\bممتاز\s+الطعم\b",
        
        r"\b(جوده|جودة|الجوده|الجودة)\b",
        r"\bمستوى\s+(الاكل|الطعام)\b",
        
        r"\b(طازج|فريش)\b",
        r"\b(مو|غير|مش)\s+طازج\b",
        r"\b(قديم|بايت)\b",
        
        r"\b(بارد|حار|محروق|يابس|ناشف)\b",
        r"\bحار\s+مره\b",
        r"\bمستوي\s+(زياده|زيادة|ناقص)\b",
        
        r"\b(دهني|دهنية|زيت)\b",
        r"\bزيت\s+كثير\b",
        r"\b(مالح|ملح)\b",
        r"\bملح\s+(زايد|زائد)\b",
        r"\b(حار|سبايسي)\s+(زياده|زيادة)\b",
        

        r"\b(تتبيل|تتبيلة|بهارات|بهار)\b",
        r"\b(صلصه|صلصة|صوص)\b",  # ✅ مع حدود الكلمات
        
        r"\b(غير|مو|مش)\s+لذيذ\b",
        r"\bطعم\s+سيء\b",
        r"\bماله\s+طعم\b"
    ],

    # 8) الإضاءة
    "الإضاءة": [
        r"\b(اضاءه|اضاءة|إضاءه|إضاءة)\b",
        r"\bاضاء(ه|ة)\s+(المكان|ضعيفه|ضعيفة|قويه|قوية|خفيفه|خفيفة|مزعجه|مزعجة)\b",
        r"\bاضاء(ه|ة)\s+(سيئه|سيئة|حلوه|حلوة|جميله|جميلة|ممتازه|ممتازة)\b",
        
        # ⚠️ استثناء كلمة "نور" لوحدها - فقط في سياقات محددة
        r"\b(الانوار|اناره|انارة)\b",  # ✅ بدون "نور" لوحدها
        r"\bانار(ه|ة)\s+(ضعيفه|ضعيفة|قويه|قوية)\b",
        
        r"\b(لمبه|لمبة|لمبات|اللمبات|سبوت|كشاف|كشافات)\b",
        
        r"\b(مظلم|ظلام|معتم)\b",
        r"\bالاضاء(ه|ة)\s+(مطفيه|مطفية|ضعيفه\s+جدا)\b",
        
    ],

    # 9) الزحمة
    "الزحمة": [
        r"\b(زحمه|زحمة|زحام|ازدحام|مزدحم|مكتظ)\b",
        r"\b(كتمه|كتمة|خانقه|خانقة)\b",
        r"\bخانق(ه|ة)\s+مره\b",
        
        r"\b(مكان|الفرع|المحل|المطعم)\s+(زحمه|زحمة)\b",
        
        r"\bازدحام\s+شديد\b",
        r"\bزحم(ه|ة)\s+(شديده|شديدة|قويه|قوية)\b",
        
        r"\b(طاولات|كراسي)\s+(قريبه|قريبة)\b",
        r"\bقريبين\s+من\s+بعض\b",
        r"\bمساح(ه|ة)\s+ضيقه\b",
        r"\b(ضيق|ضيقه|ضيقة)\b",
        
        r"\b(ما|مافي|مو)\s+في\s+جلسات\b",
        r"\bجلسات\s+قليله\b",
        r"\bاماكن\s+قليله\b",
        
        r"\bصعب\s+تلقى\s+مكان\b",
        r"\b(ما|مو)\s+(لقيت|حاصل)\s+مكان\b"
    ],
    
    # 10) الألعاب
    "الألعاب": [
        r"\b(العاب|ألعاب)\b",
        r"\bمنطق(ه|ة)\s+(العاب|ألعاب)\b",
        r"\bقسم\s+(العاب|ألعاب)\b",
        r"\bصال(ه|ة)\s+(العاب|ألعاب)\b",
        
        r"\b(ملاهي|ملاه[يى]|ترفيه|ترفيه[يى])\b",
        r"\b(أركيد|اركيد)\b",
        
        r"\b(بلايستيشن|بلاي\s*ستيشن|سوني)\b",
        r"\b(اكس\s*بوكس|نينتندو)\b",
        
        r"\b(طاول[هة]\s+بلياردو|بلياردو|بولينج|بولنج)\b",
        r"\b(هوكي|اير\s*هوكي)\b",
        
        r"\b(العاب|ألعاب)\s+(اطفال|أطفال|صغار)\b",
        r"\bمنطق(ه|ة)\s+(اطفال|أطفال)\b",
        
        r"\b(زحم(ه|ة)|ازدحام|انتظار|طابور)\s+(العاب|ألعاب)\b",
        
        r"\b(اجهز(ه|ة)|أجهزة|مكائن)\s+(العاب|ألعاب)\b",
        r"\bألعاب\s+الكترونيه\b",
        
        r"\b(تذاكر|كروت|بطاق[هة])\s+(العاب|ألعاب)\b",
        r"\bشحن\s+الكرت\b"
    ],

    # 11) الصيانة
    "الصيانة": [
        r"\b(صيان[هة]|صيانه|صيانة)\b",
        r"\bصيان(ه|ة)\s+(المكان|المحل)\b",
        r"\b(الصيانه|الصيانة|فني|فنيين|فنيون)\b",
        
        r"\b(عطل|اعطال|أعطال)\b",
        r"\b(خربان|خربانه|خربانة|مخرب)\b",
        r"\bخربان(ه|ة)?\s+مره\b",
        r"\b(مو|مش|ما)\s+شغال\b",
        r"\b(لا|ما)\s+(يعمل|يشتغل)\b",
        
        r"\b(تصليح|اصلاح|إصلاح|تصليحات|إصلاحات)\b",
        r"\b(تبديل|تغيير|استبدال)\b",
        
        r"\b(سباك[هة]|سباك|كهرباء|كهربائي|كهربائيه|كهربائية)\b",
        r"\b(تمديدات|تسريب|تهريب)\b",
        
        r"\b(مكيف|التكييف|لمبات|الاضاءه|الاضاءة|مصعد)\s+خربان\b",
        r"\bدور(ه|ة)\s+مياه\s+خربان(ه|ة)\b",
        
        r"\bصيان(ه|ة)\s+ضعيفه\b",
        r"\b(ما|مافي|مو)\s+في\s+صيان(ه|ة)\b",
        r"\b(تاخير|تأخير)\s+الصيان(ه|ة)\b"
    ],
}

# =========================
# Compile regex (important for speed)
# =========================
ASPECT_REGEX = {
    asp: [re.compile(pat) for pat in pats]
    for asp, pats in ASPECT_PATTERNS.items()
}

In [13]:
def detect_aspects_with_hits(text: str):

    if not isinstance(text, str):
        return {}

    found = {}

    for asp, regs in ASPECT_REGEX.items():
        hits = []

        for rg in regs:
            for m in rg.finditer(text):
                hits.append(m.group(0))

        if hits:
            found[asp] = sorted(set(hits))

    return found

df4["aspect_hits"] = df4["text_clean"].apply(detect_aspects_with_hits)

In [14]:
df4

,text_before,text_clean,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,aspect_hits
0,طالما فيها ال سعود الاذلاء رح تضلها بلاد ذليلة,طالما فيها ال سعود الاذلاء رح تضلها بلاد ذليلة,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,{}
1,تعال لنا ثاني مره نتشرف فيك,تعال لنا ثاني مره نتشرف فيك,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,{}
2,يامرحبا فيك عيد الزياره واجلس اكثر وروح لكذا م...,يامرحبا فيك عيد الزياره واجلس اكثر وروح لكذا م...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,{}
3,المسمار هو القرنفل وليس جوزة الطيب,المسمار هو القرنفل وليس جوزة الطيب,3,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,{}
4,اهلا اهلا فيك في بلدنا منور يا وحش,اهلا اهلا فيك في بلدنا منور يا وحش,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,{}
...,...,...,...,...,...,...,...,...,...
19434,وعليكم السلام ورحمة الله الحمد الله يبارك فيك ...,وعليكم السلام ورحمة الله الحمد الله يبارك فيك ...,5,المنطقة الغربية,المنطقة الغربية,مكة,مكة مول_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,{}
19435,بما انك دعيت هذه الدهوة اقول الله يعينك,بما انك دعيت هذه الدهوة اقول الله يعينك,1,المنطقة الغربية,المنطقة الغربية,مكة,مكة مول_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,{}
19436,انا متاكد ان سلمان سيلبي طلبك بكل سرور ولذلك و...,انا متاكد ان سلمان سيلبي طلبك بكل سرور ولذلك و...,5,المنطقة الغربية,المنطقة الغربية,مكة,مكة مول_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,{}
19437,وفقك الله اختي، نقول توكلت علي الله قال رجل لل...,وفقك الله اختي، نقول توكلت علي الله قال رجل لل...,5,المنطقة الغربية,المنطقة الغربية,مكة,مكة مول_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,{}


In [15]:
# ======================================================
# CREATE ASPECT LONG DATAFRAME (ONE ASPECT PER ROW)
# ======================================================

TEXT_COL = "text_clean"   # أو Text_TR

META_COLS = ["Stars","Region_Name", "Region_Folder", "City_Folder", "Source_File", "__path__"]
meta_existing = [c for c in META_COLS if c in df4.columns]

df4 = df4.copy()

# استخراج الجوانب
df4["aspect_hits"] = df4[TEXT_COL].apply(detect_aspects_with_hits)

# تحويل إلى صف لكل جانب
rows = []

for idx, r in df4.iterrows():

    hits_dict = r["aspect_hits"]

    if not hits_dict:
        continue

    for aspect, hits in hits_dict.items():

        rows.append({
            "row_id": idx,
            "aspect": aspect,
            "hits": " | ".join(hits),
            "text": r[TEXT_COL],
            **{c: r[c] for c in meta_existing}
        })

df_aspect_long = pd.DataFrame(rows)

print("Aspect-level rows:", len(df_aspect_long))
display(df_aspect_long.head(20))

print("\nAspect distribution:")
display(df_aspect_long["aspect"].value_counts())

Aspect-level rows: 1478


,row_id,aspect,hits,text,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__
0,12,الطعام,بهارات,الفيديو كتير حلو بس للمعلومة المسمار هوي كبش ا...,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
1,16,الانتظار,سريع | متاخر,صح وصلت متاخر بس ان شاء الله دعوة توصل سريع ال...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
2,96,الطعام,نكهه,هو صح المسمار هو نفسه القرنفل وناس تقول زر بس ...,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
3,166,الأسعار,غالية,هي غالية فعلا الا اذا تسافرها بطريقة محلية,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
4,174,الأسعار,يستاهل,يا الله هالانسان يستاهل يكون افضل رحاله,3,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
5,179,النظافة,الارض,يا عباد الله ياامة الاسلام اسالكم بالله الذي ل...,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
6,180,النظافة,الارض,يا عباد الله ياامة الاسلام اسالكم بالله الذي ل...,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
7,197,النظافة,نظيفة,هذه نيويورك العرب ه احسن منها لانها نظيفة علي ...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_06...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
8,212,الأسعار,رخيص | رخيص جدا,الرياض اغ ي من الاماكن التانيه انا عايش في الج...,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_06...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...
9,213,الطعام,الاكل,المفروض تبلع بلبن المراعي عشان الاكل يتزحلق,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_06...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...



Aspect distribution:


aspect
الأسعار           451
الطعام            270
النظافة           251
الخدمة            205
دورات المياه       80
الزحمة             75
الانتظار           64
الألعاب            33
الصيانة            24
مواقف السيارات     21
الإضاءة             4
Name: count, dtype: int64

In [16]:
aspect_overall = (
    df_aspect_long
    .groupby("aspect")
    .size()
    .reset_index(name="mentions")
    .sort_values("mentions", ascending=False)
)

display(aspect_overall)

,aspect,mentions
0,الأسعار,451
7,الطعام,270
8,النظافة,251
4,الخدمة,205
9,دورات المياه,80
5,الزحمة,75
3,الانتظار,64
1,الألعاب,33
6,الصيانة,24
10,مواقف السيارات,21


In [17]:
total_mentions = aspect_overall["mentions"].sum()

aspect_overall["percentage_%"] = (
    aspect_overall["mentions"] / total_mentions * 100
).round(2)

display(aspect_overall)

,aspect,mentions,percentage_%
0,الأسعار,451,30.51
7,الطعام,270,18.27
8,النظافة,251,16.98
4,الخدمة,205,13.87
9,دورات المياه,80,5.41
5,الزحمة,75,5.07
3,الانتظار,64,4.33
1,الألعاب,33,2.23
6,الصيانة,24,1.62
10,مواقف السيارات,21,1.42


In [18]:
aspect_unique_reviews = (
    df_aspect_long
    .groupby("aspect")["row_id"]
    .nunique()
    .reset_index(name="unique_reviews")
    .sort_values("unique_reviews", ascending=False)
)

display(aspect_unique_reviews)

,aspect,unique_reviews
0,الأسعار,451
7,الطعام,270
8,النظافة,251
4,الخدمة,205
9,دورات المياه,80
5,الزحمة,75
3,الانتظار,64
1,الألعاب,33
6,الصيانة,24
10,مواقف السيارات,21


In [19]:
aspect_by_region = (
    df_aspect_long
    .groupby(["Region_Name", "aspect"])
    .size()
    .reset_index(name="mentions")
    .sort_values(["Region_Name","mentions"], ascending=[True, False])
)

display(aspect_by_region.head(30))

,Region_Name,aspect,mentions
8,المنطقة الجنوبية,النظافة,131
0,المنطقة الجنوبية,الأسعار,126
7,المنطقة الجنوبية,الطعام,112
4,المنطقة الجنوبية,الخدمة,93
5,المنطقة الجنوبية,الزحمة,33
9,المنطقة الجنوبية,دورات المياه,29
3,المنطقة الجنوبية,الانتظار,18
1,المنطقة الجنوبية,الألعاب,14
6,المنطقة الجنوبية,الصيانة,13
10,المنطقة الجنوبية,مواقف السيارات,9


In [20]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [21]:
# =========================
# CONFIG
# =========================
MODEL_NAME = "/kaggle/input/models/aymanalzahrani7/youtube-camelbertv2/pytorch/default/1/Youtube"  # مثال شائع
MAX_LEN = 128
BATCH_SIZE = 64   # جرّب 64 على GPU، إذا حصل OOM خفّض لـ 32
SAT_MODE = "pos_only"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# (2) DEVICE + LOAD MODEL
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print("id2label:", model.config.id2label)

# Identify label indices robustly
id2label = {int(k): v for k, v in model.config.id2label.items()}
label_lower = {i: str(lab).lower() for i, lab in id2label.items()}

def _find_idx(keys):
    for i, lab in label_lower.items():
        if any(k in lab for k in keys):
            return i
    return None

pos_idx = _find_idx(["pos", "positive", "ايجاب", "إيجاب"])
neg_idx = _find_idx(["neg", "negative", "سلب", "سلبي"])

# Fallback if model uses common ordering but labels are generic like LABEL_0/1/2
if pos_idx is None or neg_idx is None:
    
    neg_idx,pos_idx =  0,1

print("Indices -> pos:", pos_idx,"neg:", neg_idx)

Device: cuda
Device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

id2label: {0: 'Negative', 1: 'Positive'}
Indices -> pos: 1 neg: 0


In [22]:
# =========================
# (3) BUILD BERT INPUT (Aspect-aware)
# =========================
# Each row: aspect [SEP] text
df_aspect_long["bert_input"] = (
    df_aspect_long["aspect"].astype(str) + " [SEP] " + df_aspect_long["text"].astype(str)
)

# Optional: drop empty
df_aspect_long = df_aspect_long[df_aspect_long["bert_input"].str.len() > 0].copy()
df_aspect_long.reset_index(drop=True, inplace=True)

print("After bert_input:", df_aspect_long.shape)


After bert_input: (1478, 11)


In [23]:
# =========================
# (4) BATCH INFERENCE FUNCTION (BINARY)
# =========================
def batch_predict_binary(texts, batch_size=64, max_len=128, log_every_batches=300):
    """
    Returns:
      p_pos, p_neg (np arrays length N)
    """
    all_pos, all_neg = [], []

    N = len(texts)

    for b, start in enumerate(range(0, N, batch_size), 1):
        batch_texts = texts[start:start + batch_size]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            logits = model(**enc).logits   # [B, 2]

        probs = F.softmax(logits, dim=1)   # [B, 2]
        probs = probs.detach().cpu().numpy()

        all_pos.append(probs[:, pos_idx])
        all_neg.append(probs[:, neg_idx])

        if (b % log_every_batches) == 0:
            done = min(start + batch_size, N)
            print(f"Processed {done}/{N} rows...")

    p_pos = np.concatenate(all_pos, axis=0)
    p_neg = np.concatenate(all_neg, axis=0)

    return p_pos, p_neg


# =========================
# (5) RUN INFERENCE
# =========================
texts = df_aspect_long["bert_input"].tolist()

p_pos, p_neg = batch_predict_binary(
    texts,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN,
    log_every_batches=300
)

df_aspect_long["p_positive"] = p_pos
df_aspect_long["p_negative"] = p_neg


# Final label by argmax
# stack order = [neg, pos]
pred_idx = np.argmax(
    np.stack([
        df_aspect_long["p_negative"],
        df_aspect_long["p_positive"]
    ], axis=1),
    axis=1
)

# idx 0 = negative, idx 1 = positive
df_aspect_long["sentiment"] = np.where(
    pred_idx == 1,
    "ايجابي",
    "سلبي"
)


# Satisfaction score
# In binary setup, usually positive probability is enough
df_aspect_long["satisfaction_score"] = df_aspect_long["p_positive"]


print("Inference done.")
display(df_aspect_long.head(10))

Inference done.


,row_id,aspect,hits,text,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,bert_input,p_positive,p_negative,sentiment,satisfaction_score
0,12,الطعام,بهارات,الفيديو كتير حلو بس للمعلومة المسمار هوي كبش ا...,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,الطعام [SEP] الفيديو كتير حلو بس للمعلومة المس...,0.003727,0.996273,سلبي,0.003727
1,16,الانتظار,سريع | متاخر,صح وصلت متاخر بس ان شاء الله دعوة توصل سريع ال...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,الانتظار [SEP] صح وصلت متاخر بس ان شاء الله دع...,0.999968,0.000032,ايجابي,0.999968
2,96,الطعام,نكهه,هو صح المسمار هو نفسه القرنفل وناس تقول زر بس ...,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,الطعام [SEP] هو صح المسمار هو نفسه القرنفل ونا...,0.000040,0.999960,سلبي,0.000040
3,166,الأسعار,غالية,هي غالية فعلا الا اذا تسافرها بطريقة محلية,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,الأسعار [SEP] هي غالية فعلا الا اذا تسافرها بط...,0.999931,0.000069,ايجابي,0.999931
4,174,الأسعار,يستاهل,يا الله هالانسان يستاهل يكون افضل رحاله,3,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,الأسعار [SEP] يا الله هالانسان يستاهل يكون افض...,0.999968,0.000032,ايجابي,0.999968
5,179,النظافة,الارض,يا عباد الله ياامة الاسلام اسالكم بالله الذي ل...,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,النظافة [SEP] يا عباد الله ياامة الاسلام اسالك...,0.000104,0.999896,سلبي,0.000104
6,180,النظافة,الارض,يا عباد الله ياامة الاسلام اسالكم بالله الذي ل...,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_05...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,النظافة [SEP] يا عباد الله ياامة الاسلام اسالك...,0.000104,0.999896,سلبي,0.000104
7,197,النظافة,نظيفة,هذه نيويورك العرب ه احسن منها لانها نظيفة علي ...,5,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_06...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,النظافة [SEP] هذه نيويورك العرب ه احسن منها لا...,0.999934,0.000066,ايجابي,0.999934
8,212,الأسعار,رخيص | رخيص جدا,الرياض اغ ي من الاماكن التانيه انا عايش في الج...,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_06...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,الأسعار [SEP] الرياض اغ ي من الاماكن التانيه ا...,0.000036,0.999964,سلبي,0.000036
9,213,الطعام,الاكل,المفروض تبلع بلبن المراعي عشان الاكل يتزحلق,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_youtube-comments-scraper_2025-11-11_06...,/kaggle/input/datasets/ziyadaltalhi/youtube-da...,الطعام [SEP] المفروض تبلع بلبن المراعي عشان ال...,0.000069,0.999931,سلبي,0.000069


In [24]:
# احفظ التفاصيل (كل صف = جانب داخل تعليق + احتمالات)
df_aspect_long.drop(columns=["bert_input"], errors="ignore").to_csv(
    "/kaggle/working/aspect_sentiment_detail-youtube.csv",
    index=False,
    encoding="utf-8-sig"
)



print("Saved to /kaggle/working/")

Saved to /kaggle/working/
